<a href="https://colab.research.google.com/github/averoe/ML_Practice/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/averoe/ML_Practice/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
import pandas as pd
from google.colab import userdata
from datasets import load_dataset

REPO_ID = "FlyRank/internship-warehouse"
token = userdata.get('HF_TOKEN')

try:
    print("Fetching dimension data (dim_content)... updates 'df'")
    dataset = load_dataset(REPO_ID, data_files="dim_content.parquet", token=token, split="train")
    df = dataset.select(range(min(100000, len(dataset)))).to_pandas()

    print("Fetching performance data (fact_content_daily_performance)... updates 'fact_df'")
    fact_dataset = load_dataset(REPO_ID, "fact_content_daily_performance", token=token, split="train")
    fact_df = fact_dataset.select(range(min(1000000, len(fact_dataset)))).to_pandas()

    print(f"Data loaded. Content: {len(df)} rows, Performance: {len(fact_df)} rows.")
except Exception as e:
    print(f"Error fetching data: {e}")

Fetching dimension data (dim_content)... updates 'df'
Fetching performance data (fact_content_daily_performance)... updates 'fact_df'


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  624kB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 3.22MB            

...

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  146MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

Data loaded. Content: 100000 rows, Performance: 1000000 rows.


## 1. My lane (or freestyle) and why

**Lane 2: Refresh / Content Opportunity Scoring**

I’ve chosen Lane 2 because it addresses a common challenge in large-scale search: content decay. In a massive warehouse, identifying which legacy pages still have search 'pull' but are starting to fade allows us to act before traffic is lost. By focusing on high-visibility but stale pages, we ensure that editorial resources are spent where they can protect existing revenue and user engagement.

In [19]:
imp_col_fact = next((c for c in fact_df.columns if 'impressions' in c.lower()), None)

if not imp_col_fact:
    print(f"Columns found in fact_df: {list(fact_df.columns)}")
    raise KeyError("Could not find an impressions column in fact_df")

print(f"Aggregating using column: {imp_col_fact}")

perf_agg = fact_df.groupby('content_hash_id')[imp_col_fact].sum().reset_index()
perf_agg.rename(columns={imp_col_fact: 'impressions_last_90d'}, inplace=True)

if 'impressions_last_90d' in df.columns:
    df = df.drop(columns=['impressions_last_90d'])
df = df.merge(perf_agg, on='content_hash_id', how='left').fillna({'impressions_last_90d': 0})

df['content_created_date'] = pd.to_datetime(df['content_created_date'])
df['content_age_days'] = (pd.Timestamp.now() - df['content_created_date']).dt.days

stale_visible = df[(df['impressions_last_90d'] >= 500) & (df['content_age_days'] >= 180)]

print(f"Lane 2 Analysis Complete.")
print(f"Stale Visible Candidates: {len(stale_visible)}")
display(stale_visible[['content_hash_id', 'impressions_last_90d', 'content_age_days']].head())

Aggregating using column: gsc_impressions
Lane 2 Analysis Complete.
Stale Visible Candidates: 1595


,content_hash_id,impressions_last_90d,content_age_days
30002,content_7460bf40a35b8e21,858.0,521
30005,content_7468d96a992d9939,1827.0,502
30007,content_74728e96f00f02e9,1304.0,480
30009,content_7476254360f720e6,4111.0,502
30015,content_74889431110faa21,830.0,456


## 2. The question: decision, action, cost of a wrong call

**Decision:** Which content pages should a limited team of editors refresh first?

**Action:** Human review of the top-ranked pages to decide between rewriting, updating metadata, or pruning.

**Cost of a wrong call:** Low-risk but inefficient. A 'false positive' (recommending a page that didn't need it) wastes editorial time. A 'false negative' (missing a declining page) leads to avoidable traffic loss and revenue decay.

In [20]:
total_pages = len(df)
visible_stale_count = len(stale_visible)

print(f"Total inventory: {total_pages} items")
print(f"High-volume stale items (candidates): {visible_stale_count}")
print(f"Opportunity percentage: {(visible_stale_count / total_pages):.2%}")

Total inventory: 100000 items
High-volume stale items (candidates): 1595
Opportunity percentage: 1.59%


## 3. Quick look at the data (2-3 real numbers)

After analyzing a slice of 100,000 pages from the Hugging Face warehouse release, here is what the data shows:
- **1.59%** of the inventory (1,595 pages) are 'Stale Visible' candidates—mature content (180+ days old) that still pulls at least 500 impressions.
- **2.99%** of pages show an active performance decline when comparing the last 30 days against the previous 30.
- **0.191 Correlation:** There is a measurable positive correlation between a page's age and its likelihood of declining, confirming that 'staleness' is a relevant signal for this lane.

In [22]:
import numpy as np

fact_df['report_date'] = pd.to_datetime(fact_df['report_date'])
max_date = fact_df['report_date'].max()

recent_window = fact_df[fact_df['report_date'] > (max_date - pd.Timedelta(days=30))]
prev_window = fact_df[(fact_df['report_date'] <= (max_date - pd.Timedelta(days=30))) &
                      (fact_df['report_date'] > (max_date - pd.Timedelta(days=60)))]

recent_imps = recent_window.groupby('content_hash_id')[imp_col_fact].sum()
prev_imps = prev_window.groupby('content_hash_id')[imp_col_fact].sum()

trend_df = pd.DataFrame({'recent': recent_imps, 'prev': prev_imps}).fillna(0)
trend_df['is_down'] = (trend_df['recent'] < trend_df['prev']).astype(int)

if 'is_down' in df.columns:
    df = df.drop(columns=['is_down'])
df = df.merge(trend_df[['is_down']], on='content_hash_id', how='left').fillna({'is_down': 0})

decline_pct = df['is_down'].mean()
age_corr = df[['content_age_days', 'is_down']].corr().iloc[0, 1]

print(f"Percentage of pages declining: {decline_pct:.2%}")
print(f"Correlation (Age vs Decline): {age_corr:.3f}")

Percentage of pages declining: 2.99%
Correlation (Age vs Decline): 0.191


## 4. Careful words: what I can and can't claim

**What I can claim:** I can provide an evidence-backed ranking system to help editors prioritize their workflow. I have measured that roughly **3%** of our high-volume content is in a downward trend and confirmed that older pages are statistically more likely to experience this decay. This allows us to move from 'guessing' what to update to a **data-driven decision support** model.

**What I can't claim:** I can't claim that refreshing these pages will automatically fix the decline, as search rankings are influenced by many external factors I can't control (like competitor moves or algorithm updates). This is a tool for prioritization, not a guaranteed growth hack. The relationship between age and decay is directional, not a absolute law.

In [23]:
import pandas as pd

forbidden_cols = ['health_score', 'priority_score', 'action_type', 'refresh_tier']
found_forbidden = [c for c in forbidden_cols if c in df.columns]

if not found_forbidden:
    print("Signal Audit Passed: No product decision flags found. Using observable signals only.")
else:
    print(f"Warning: Found internal product flags: {found_forbidden}")

decline_count = df['is_down'].sum()
total_count = len(df)
actual_decline_pct = (decline_count / total_count) * 100

print(f"Total items in slice: {total_count}")
print(f"Total labels for decision support: {int(decline_count)} 'down' pages identified.")
print(f"Verified decline percentage: {actual_decline_pct:.2f}%")

Signal Audit Passed: No product decision flags found. Using observable signals only.
Total items in slice: 100000
Total labels for decision support: 2989 'down' pages identified.
Verified decline percentage: 2.99%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.